# Agon: Eos, diagnosis (why they die, how well they learn, what they can do)

This notebook trains nothing. It takes a **trained run** (from Drive), lives its agents in the run's own world, and answers four questions with numbers:

| question | section | how |
|---|---|---|
| How well are they learning, and is it still moving? | 2 | `history.json`: last third against the third before, per metric |
| Do their choices follow their needs? | 3 | `decision_check.py`: trained vs untrained vs random |
| **Why do they die?** (every cause) | 4 | the cause table, then forensics for thirst, starvation, predation and cold, a brain probe, and a counterfactual shelter rule |
| **What do they do while a need is low?** | 4 | beside water / holding food, do they drink / eat? far from it, do they go? |
| What can they do and what do they never do? | 5 | the fire ladder and every action family, against random agents |

**Run the sections in order. Each costs minutes, not hours.** The expensive part is section 4 (3 seeds x 80 days, four kinds of agent); set `DAYS` and `SEEDS` smaller for a first look.

**Read the cause table first.** It says what actually kills them. In the one run diagnosed so far (13M steps, predator-heavy variant) cold killed 1 of 24; thirst and predation did the rest, so the cold forensics below are only worth reading when cold is a real share of the deaths.

**What the cold test separates.** Every cold death lands in exactly one bin:

| bin | what it says | what you would do next |
|---|---|---|
| froze inside shelter | shelter is not enough: cold snaps, snow, no neighbour, thin insulation, children | warmth is a body/world problem: fire, clothing, huddling, not a decision problem |
| knew shelter within 12 cells, stayed out | a decision failure with the remedy known | credit assignment or timing: the penalty comes hours later |
| shelter within 12 cells they had never visited | exploration or memory | exploration pressure, longer sight, a map |
| no shelter within 12 cells | placement, a long walk | the world or spawn, not the brain |

The probe asks the brain directly: with body temperature set to 0, -0.3 ... -0.9 (nothing else changed), does the critic's value fall (it knows cold is costly), and does the chance of seeking shelter or resting rise (it knows what to do)? The reflex test lays one rule over the same agents ("when cold, or from two hours before dark, go to shelter if not in it") and counts the cold deaths again. If they vanish, the world is survivable and the missing piece is the agents' decision; if they stay, something else is killing them.

**What this does not prove.** One run, a few seeds: differences of a handful of deaths are noise. The probe changes one number for one tick, so it understates a sensitivity built over hours. "Visited" means cells the agent has stood on or near (`State.visited`), not everything it remembers.

## 1. GPU and code

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: Runtime > Change runtime type > A100 GPU, then restart."

Upload `lifesim_code.zip` (the commit the run was trained on if you can; the network must have the same shape, so a run from a much older or newer commit may refuse to load), or set `ZIP_FROM_DRIVE`.

In [ ]:
import glob, html, json, shutil, subprocess, sys, zipfile
ZIP_FROM_DRIVE = ""   # e.g. "/content/drive/MyDrive/agon/lifesim_code.zip"
if ZIP_FROM_DRIVE and os.path.exists(ZIP_FROM_DRIVE):
    zip_path = ZIP_FROM_DRIVE
else:
    from google.colab import files
    zip_path = next(iter(files.upload()))
shutil.rmtree("/content/lifesim", ignore_errors=True)
os.makedirs("/content/lifesim")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/lifesim")
if not os.path.exists("/content/lifesim/pyproject.toml"):
    inner = next(os.path.dirname(p) for p in glob.glob("/content/lifesim/*/pyproject.toml"))
    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), "/content/lifesim")
os.chdir("/content/lifesim")
!pip install -q -U "flax>=0.12.10" "optax>=0.2.8" pytest
!pip install -q --no-deps -e .
sys.path.insert(0, "/content/lifesim")
from lifesim.presets import code_version
from lifesim import env as E
assert os.path.exists("scripts/diagnose.py"), "This zip has no scripts/diagnose.py: upload the zip that comes with this notebook."
print("code version", code_version(), "| observation channels", E.N_OBS_CHANNELS)

## 2. Pick the run

Lists every trained run on Drive. Set `RUN` to the one to diagnose (the baseline is `.../validate/caveman`). The cell prints the settings it was trained with, how far it got, and whether its weights are there.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
found = sorted(glob.glob("/content/drive/MyDrive/agon/runs/*/validate/*/history.json"))
for f in found:
    h = json.load(open(f))
    last = h["history"][-1] if h["history"] else {}
    print(os.path.dirname(f), "|", f"{last.get('agent_steps', 0) / 1e6:.0f}M steps")

In [ ]:
RUN = ""   # paste one of the folders printed above, e.g. ".../intel_xxxx_society30_w4/validate/caveman"
assert RUN and os.path.exists(f"{RUN}/history.json"), "Set RUN to a folder printed above."
H = json.load(open(f"{RUN}/history.json"))
a = H["args"]
print({k: a.get(k) for k in ("slate", "world", "agents", "grid", "world_model", "reproduction", "retention_coef", "entropy_target",
                              "reward_mode", "days_per_year", "tick_minutes", "hidden", "encoder") if k in a})
print("rows:", len(H["history"]), "| final weights:", os.path.exists(f"{RUN}/params.msgpack"),
      "| checkpoint:", os.path.exists(f"{RUN}/checkpoint.pkl"))
assert os.path.exists(f"{RUN}/params.msgpack") or os.path.exists(f"{RUN}/checkpoint.pkl"), "No weights in this run."
OUT = f"{RUN}/diagnose"
os.makedirs(OUT, exist_ok=True)

## 3. How well are they learning

Each metric is averaged over the first, middle and last third of the log. **A verdict is shown only where higher or lower is clearly better**; the rest are context. "Still improving" means the last third beat the middle one by more than 5%.

Read the entropy rows with care: runs made before commit `ab25481` logged entropy and its bonus about 10x too low, so their absolute values mean nothing (the real entropy is about 1 to 2 nats). Compare them only within the run.

In [ ]:
!python scripts/diagnose.py --run {RUN} --sections learn --out {OUT}

In [ ]:
import matplotlib.pyplot as plt
rows = H["history"]
x = [r["agent_steps"] / 1e6 for r in rows]
def series(path):
    out = []
    for r in rows:
        v = r
        for p in path.split("."):
            v = v.get(p) if isinstance(v, dict) else None
        out.append(v)
    return out
fig, ax = plt.subplots(3, 3, figsize=(15, 10))
def line(a, paths, title):
    for p in paths:
        s = series(p)
        a.plot([xi for xi, v in zip(x, s) if v is not None], [v for v in s if v is not None], label=p.split(".")[-1])
    a.set_title(title); a.legend(fontsize=7); a.set_xlabel("M agent-steps")
line(ax[0, 0], ["survival_score_per_world_day"], "survival score per world-day")
line(ax[0, 1], ["mean_age_at_death"], "mean age at death (ticks)")
causes = sorted({c for r in rows for c in r["deaths"]})
ax[0, 2].stackplot(x, [[r["deaths"].get(c, 0) for r in rows] for c in causes], labels=causes); ax[0, 2].legend(fontsize=6, ncol=2); ax[0, 2].set_title("deaths by cause per log row")
split = ["open_adult", "open_child", "sheltered_adult", "sheltered_child"]
ax[1, 0].stackplot(x, [[r["cold_deaths"].get(k, 0) for r in rows] for k in split], labels=split); ax[1, 0].legend(fontsize=7); ax[1, 0].set_title("cold deaths: where and who")
line(ax[1, 1], ["winter_huddle_share"], "huddling on winter nights")
line(ax[1, 2], ["entropy", "entropy_coef"], "entropy and its bonus (see the logging note)")
line(ax[2, 0], ["world_model.trust_in_ideas"], "trust in its ideas")
line(ax[2, 1], ["fire_ladder_per_1000.near_fire", "fire_ladder_per_1000.fed_fire", "fire_ladder_per_1000.pulled_burning_wood"], "fire ladder per 1000 agent-ticks")
line(ax[2, 2], ["imitation_loss", "watched_fraction", "learned_share"], "social learning and sleep")
plt.tight_layout(); plt.show()

## 4. Do their choices follow their needs

The same check as before (`decision_check.py`), for this run against untrained and random agents: 10 days, 3 seeds. A rule with too few ticks behind it prints n/a. **Lift** is the trained rate over the random rate: 1.0 is chance.

In [ ]:
!python scripts/decision_check.py --run {RUN} --days 10 --seeds 3 --out {OUT}/decision_check.json

## 5. Why do they die

One command, four kinds of agent: **random**, the **trained** agents, and the trained agents with the **cold reflex** and the **dusk reflex** laid over them. It prints, in order: survival, the cold-death forensics, the brain probe, the fire ladder and action inventory, what they choose while thirsty or hungry, and the forensics for every cause of death (thirst, starvation, predation). Results are saved so the next cells (and a re-run with `--reuse`) do not repeat the rollouts.

Set `DAYS` to at least two game years (the run's year is printed in the header) so winter happens twice. More seeds make the bins steadier.

In [ ]:
DAYS, SEEDS = 80, 3
!python scripts/diagnose.py --run {RUN} --days {DAYS} --seeds {SEEDS} --sections survival,cold,probe,fire,caps,needs,deaths --out {OUT}

In [ ]:
import numpy as np
D = json.load(open(f"{OUT}/diagnose.json"))
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
sv = D.get("survival", {})
names = [k for k in ("random", "policy", "reflex_cold", "reflex_dusk") if k in sv]
ay = [sv[k]["agent_years"] for k in names]
ax[0].bar(names, [sv[k]["deaths"].get("cold", 0) / max(sv[k]["agent_years"], 1e-9) for k in names])
ax[0].set_title("cold deaths per agent-year"); ax[0].tick_params(axis="x", labelrotation=20)
cr = D.get("cold_rows", [])
if cr:
    ax[1].hist([x["exposure_hours"] for x in cr], bins=20); ax[1].set_title("hours of exposure before death (body < -0.3)")
    ax[2].scatter([x["d_known"] for x in cr], [x["d_true"] for x in cr], c=[x["frac_in_cover"] for x in cr], cmap="viridis")
    ax[2].set_xlabel("cells to shelter they had visited (13 = none within 12)"); ax[2].set_ylabel("cells to any shelter")
    ax[2].set_title("where was shelter when the exposure began (color: share of the exposure in shelter)")
plt.tight_layout(); plt.show()
pr = D.get("probe", {})
fig, ax = plt.subplots(1, max(len(pr), 1), figsize=(5 * max(len(pr), 1), 3.8), squeeze=False)
temps = [0.0, -0.3, -0.5, -0.7, -0.9]
for a_, (g, v) in zip(ax[0], pr.items()):
    t = np.array(v["table"])
    a_.plot(temps, t[:, 0], "k-o", label="critic value")
    a_.set_title(f"{g} (n={v['n']})", fontsize=9); a_.set_xlabel("body temperature set to"); a_.invert_xaxis()
    b = a_.twinx()
    for j, nm in ((1, "P(seek_shelter)"), (2, "P(rest/sleep)"), (3, "P(move)")):
        b.plot(temps, t[:, j], "--o", label=nm)
    b.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

### How to read it

1. **Section 5, survival table.** `reflex_cold` and `reflex_dusk` against `policy`: if cold deaths fall to near zero, the world can be survived by *being in shelter*, and the agents' failing is the decision or its timing. If they fall only for `reflex_dusk`, the penalty comes too late for a reactive rule and the lesson needs planning (this is the world model's job, and it is unused). If neither helps, shelter is not the missing piece.
2. **Cold forensics.** Use the bin table (above). Adults and children are counted separately in the narrative line; check the child share before blaming the policy.
3. **Probe.** Critic flat across temperatures = the value of warmth was never learned (credit assignment). Value falls but P(seek_shelter) flat outside shelter at night = it feels the danger and has no habit of the remedy. Both respond = perception and remedy exist, the failure is elsewhere (timing, distance, competing needs).
4. **Fire.** A rung at or below random is a rung never learned. The `cold` columns say whether they reach for fire when it matters.
5. **Capabilities.** NEVER = not once in the rollouts. `<= chance` = no better than random agents. (`relieve` needs excretion, `delve` underground and `write` writing; if the run did not turn those on, NEVER is expected.)
6. **While a need is low.** Thirst: beside water and not drinking = a decision failure; far from water and not heading there = navigation or memory; none within 12 cells = placement. Hunger: holding food and not eating = decision; empty-handed on good forage and not grasping = the gather step. Compare each row with the random agents' row.
7. **Deaths by cause.** The table is the ranking of what to fix. Thirst: `reached water while low, died anyway` means the resource was there and the need was not met. Starvation: `held food while starving, died anyway` is the same for food. Predation: `ambush` cannot be avoided by choice; `never chose FLEE` can. Counts of a handful are anecdotes: use more seeds or days before acting on a bin.

## 6. Look at the deaths themselves

The continuous world (400 days) writes an event log with causes. These cells read it from Drive. Skip them if the run has no `continuous/` folder.

In [ ]:
CONT = f"{RUN}/continuous"
print(os.path.exists(CONT), sorted(os.listdir(CONT))[:12] if os.path.exists(CONT) else "")
!python scripts/causal_story.py {CONT} --max 40 2>&1 | tail -60

In [ ]:
# Cold moments in the decisions: what the options were, the probabilities, what the choice leaned on.
CHUNKS = sorted(glob.glob(f"{CONT}/chunk_*.html"))
print([os.path.basename(c) for c in CHUNKS])
if CHUNKS:
    !python scripts/show_decisions.py {CHUNKS[0]} {CHUNKS[-1]} --tags cold,hurt --max 8 2>&1 | tail -120

## 7. What to paste back

1. Section 3's table and section 5's output (survival, the cause table and its forensics, probe, fire, capabilities, needs). Everything is also in `{OUT}/diagnose.json` on Drive.
2. Section 4's decision check.
3. Which run you diagnosed and which commit the zip is from.

**Then the next experiment follows from the bins**, not from a guess: shelter-before-night is a planning problem (world model, longer horizon, a shaping signal for being warm), fire discovery is an exploration problem (a curriculum or an easier ladder), a probe with flat value is a credit-assignment problem.